# Baseline Colab — MOT15 + MOT16

Генерация `.npy` (det + ReID) и прогон оригинального DeepSORT на **6 sequences**.  
Оба набора генерируются одинаково через `generate_detections.py` → `MOT15_train/` и `MOT16_train/`.

In [1]:
DRIVE_RESOURCES = "/content/drive/MyDrive/DeepSORT/resources"

from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.isdir(DRIVE_RESOURCES), f"Папка не найдена: {DRIVE_RESOURCES}"

# Клонируем код (HTTPS, без SSH)
if not os.path.isdir("/content/DeepSORT_Project_CV"):
    !git clone https://github.com/Valeriia-Reznik-Dev/DeepSORT_Project_CV.git /content/DeepSORT_Project_CV

%cd /content/DeepSORT_Project_CV
!git pull

# Подключаем ваши данные с Drive
if os.path.islink("resources"):
    os.remove("resources")
elif os.path.isdir("resources"):
    !rm -rf resources
!ln -s "{DRIVE_RESOURCES}" resources

print("resources ->", os.path.realpath("resources"))

Mounted at /content/drive
Cloning into '/content/DeepSORT_Project_CV'...
remote: Enumerating objects: 493, done.
remote: Counting objects: 100% (493/493), done.
remote: Compressing objects: 100% (261/261), done.
remote: Total 493 (delta 284), reused 418 (delta 209), pack-reused 0 (from 0)
Receiving objects: 100% (493/493), 192.34 KiB | 1.49 MiB/s, done.
Resolving deltas: 100% (284/284), done.
/content/DeepSORT_Project_CV
Already up to date.
resources -> /content/drive/MyDrive/DeepSORT/resources


In [2]:
!pip install -q "numpy>=2.0" opencv-python scipy pyyaml

import numpy as np
print("NumPy", np.__version__)
assert np.__version__.split(".")[0] >= "2", (
    "numpy<2 обнаружен. Перезапустите рантайм (Runtime -> Restart session) и выполните ячейку заново."
)

import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU", tf.config.list_physical_devices("GPU"))


NumPy 2.0.2
TensorFlow 2.20.0
GPU [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
import os

MOT15_SEQUENCES = ["TUD-Campus", "TUD-Stadtmitte", "KITTI-17", "PETS09-S2L1"]
MOT16_SEQUENCES = ["MOT16-09", "MOT16-11"]
DRIVE_RESOURCES = "/content/drive/MyDrive/DeepSORT/resources"

paths = {
    "model": os.path.join(DRIVE_RESOURCES, "networks", "mars-small128.pb"),
    "mot15_dir": "resources/detections/MOT15/train",
    "mot16_dir": "resources/detections/MOT16/train",
    "mot15_npy_dir": "resources/detections/MOT15_train",
    "mot16_npy_dir": "resources/detections/MOT16_train",
}

assert os.path.isfile(paths["model"]), f"Missing: {paths['model']}"

for seq in MOT15_SEQUENCES:
    assert os.path.isdir(os.path.join(paths["mot15_dir"], seq)), f"Missing MOT15 video: {seq}"
for seq in MOT16_SEQUENCES:
    assert os.path.isdir(os.path.join(paths["mot16_dir"], seq)), f"Missing MOT16 video: {seq}"

print("OK: model + MOT15/MOT16 videos on Drive")

OK: model + MOT15/MOT16 videos on Drive


In [4]:
import os
import time
from google.colab import drive

DRIVE_ROOT = "/content/drive"
DRIVE_RESOURCES = "/content/drive/MyDrive/DeepSORT/resources"
SRC_MODEL = os.path.join(DRIVE_RESOURCES, "networks", "mars-small128.pb")
MODEL = "/content/mars-small128.pb"


def ensure_drive() -> None:
    """Remount Drive if the FUSE mount is dead (OSError 107)."""
    try:
        with open(SRC_MODEL, "rb") as f:
            f.read(1)
    except OSError as exc:
        print(f"Drive mount unhealthy ({exc}); remounting ...")
        drive.mount(DRIVE_ROOT, force_remount=True)


def copy_from_drive(src: str, dst: str, *, chunk_size: int = 1024 * 1024, retries: int = 5) -> None:
    """Copy a Drive file to local disk; remount Drive on FUSE disconnect (errno 107)."""
    ensure_drive()
    src_size = os.path.getsize(src)
    if os.path.isfile(dst) and os.path.getsize(dst) == src_size:
        print(f"Already cached: {dst}")
        return

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
                while True:
                    chunk = fsrc.read(chunk_size)
                    if not chunk:
                        break
                    fdst.write(chunk)
            if os.path.getsize(dst) == src_size:
                print(f"Copied {src_size / 1e6:.1f} MB -> {dst}")
                return
            last_err = OSError(f"size mismatch: got {os.path.getsize(dst)}, expected {src_size}")
        except OSError as exc:
            last_err = exc
            if os.path.isfile(dst):
                os.remove(dst)
            print(f"Drive copy failed ({exc}); remount + retry {attempt}/{retries} in 5s ...")
            time.sleep(5)
            ensure_drive()
            src_size = os.path.getsize(src)

    raise OSError(
        f"Could not copy {src} -> {dst}: {last_err}. "
        "Remount Drive: Runtime -> Restart session, rerun from cell 1, "
        "or drive.mount('/content/drive', force_remount=True)."
    )


copy_from_drive(SRC_MODEL, MODEL)
print("MODEL ->", MODEL)

# Кадры ниже читаются через симлинк resources -> Drive; убедимся, что монтирование живо.
ensure_drive()

# MOT15 + MOT16: один pipeline, generate_detections.py
!python tools/generate_detections.py \
    --model="{MODEL}" \
    --mot_dir=resources/detections/MOT15/train \
    --output_dir=resources/detections/MOT15_train

!python tools/generate_detections.py \
    --model="{MODEL}" \
    --mot_dir=resources/detections/MOT16/train \
    --output_dir=resources/detections/MOT16_train

Streaming output truncated to the last 5000 lines.
Frame 00323/00900
Frame 00324/00900
Frame 00325/00900
Frame 00326/00900
Frame 00327/00900
Frame 00328/00900
Frame 00329/00900
Frame 00330/00900
Frame 00331/00900
Frame 00332/00900
Frame 00333/00900
Frame 00334/00900
Frame 00335/00900
Frame 00336/00900
Frame 00337/00900
Frame 00338/00900
Frame 00339/00900
Frame 00340/00900
Frame 00341/00900
Frame 00342/00900
Frame 00343/00900
Frame 00344/00900
Frame 00345/00900
Frame 00346/00900
Frame 00347/00900
Frame 00348/00900
Frame 00349/00900
Frame 00350/00900
Frame 00351/00900
Frame 00352/00900
Frame 00353/00900
Frame 00354/00900
Frame 00355/00900
Frame 00356/00900
Frame 00357/00900
Frame 00358/00900
Frame 00359/00900
Frame 00360/00900
Frame 00361/00900
Frame 00362/00900
Frame 00363/00900
Frame 00364/00900
Frame 00365/00900
Frame 00366/00900
Frame 00367/00900
Frame 00368/00900
Frame 00369/00900
Frame 00370/00900
Frame 00371/00900
Frame 00372/00900
Frame 00373/00900
Frame 00374/00900
Frame 00375/0

In [5]:
import numpy as np

ALL_SEQUENCES = [
    (paths["mot15_npy_dir"], MOT15_SEQUENCES),
    (paths["mot16_npy_dir"], MOT16_SEQUENCES),
]

for npy_dir, seqs in ALL_SEQUENCES:
    print(f"--- {npy_dir}")
    for seq in seqs:
        path = os.path.join(npy_dir, f"{seq}.npy")
        assert os.path.isfile(path), f"Missing: {path}"
        print(seq, np.load(path).shape)

print("OK: all 6 .npy ready")

--- resources/detections/MOT15_train
TUD-Campus (322, 138)
TUD-Stadtmitte (1129, 138)
KITTI-17 (552, 138)
PETS09-S2L1 (5578, 138)
--- resources/detections/MOT16_train
MOT16-09 (5976, 138)
MOT16-11 (8590, 138)
OK: all 6 .npy ready


In [6]:
!python scripts/run_baseline.py


Running TUD-Campus
Processing frame 00001
Processing frame 00002
Processing frame 00003
Processing frame 00004
Processing frame 00005
Processing frame 00006
Processing frame 00007
Processing frame 00008
Processing frame 00009
Processing frame 00010
Processing frame 00011
Processing frame 00012
Processing frame 00013
Processing frame 00014
Processing frame 00015
Processing frame 00016
Processing frame 00017
Processing frame 00018
Processing frame 00019
Processing frame 00020
Processing frame 00021
Processing frame 00022
Processing frame 00023
Processing frame 00024
Processing frame 00025
Processing frame 00026
Processing frame 00027
Processing frame 00028
Processing frame 00029
Processing frame 00030
Processing frame 00031
Processing frame 00032
Processing frame 00033
Processing frame 00034
Processing frame 00035
Processing frame 00036
Processing frame 00037
Processing frame 00038
Processing frame 00039
Processing frame 00040
Processing frame 00041
Processing frame 00042
Processing fram

In [7]:
# Этап 2: TrackEval HOTA на baseline results
!pip install -q "trackeval==1.3.0"

!python scripts/run_eval.py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.7/207.7 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 94.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.0 which is incompatible.

CLEAR Config:
METRICS              : ['HOTA', 'CLEAR', 'Identity'] 
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
METRICS              : ['HOTA', 'CLEAR', 'Identity'] 
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Evaluating 1 tracker(s) on 4 sequence(s) for 1 class(es) on

## Detector F1 (YOLO + NanoDet + MMDet)

Сравнение детекций с GT: Precision / Recall / F1 для **всех трёх** детекторов.  
Сначала smoke test (`--max-frames 30`), потом полный прогон без флага.

In [8]:
!python scripts/setup_detectors_colab.py
# веса NanoDet + RTMDet (если ещё нет в resources/models/)
!python scripts/download_detector_models.py

# Полный прогон: Precision / Recall / F1 + FPS по всем 6 видео.
!python scripts/run_detector_eval.py --detector yolo nanodet mmdet

> /usr/bin/python3 -m pip install -q ultralytics scikit-learn
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.9 MB/s eta 0:00:00
> git clone -b v1.0.0-alpha-1 --depth 1 https://github.com/RangiLyu/nanodet.git /content/DeepSORT_Project_CV/third_party/nanodet
Cloning into '/content/DeepSORT_Project_CV/third_party/nanodet'...
remote: Enumerating objects: 312, done.
remote: Counting objects: 100% (312/312), done.
remote: Compressing objects: 100% (279/279), done.
remote: Total 312 (delta 48), reused 158 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (312/312), 1.40 MiB | 4.67 MiB/s, done.
Resolving deltas: 100% (48/48), done.
Note: switching to '08bad3294608953b1e1a8aacc50f67336420a435'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to

## ReID clustering metrics (4 models, 2 sources)

Standalone-оценка на **GT pedestrian crops**: Silhouette, Calinski–Harabasz, Fowlkes–Mallows.

| Модель | Источник |
|--------|----------|
| `osnet`, `resnet50` | torchreid |
| `resnet50_ibn`, `fastreid` | fast-reid |

Сначала smoke test (`--max-frames 30 --max-samples 500`), потом полный прогон.

In [9]:
!git pull
!python scripts/setup_reid_colab.py
!python scripts/download_reid_models.py

# smoke test — все 4 ReID модели (2 источника)
!python scripts/run_reid_eval.py --model osnet resnet50 resnet50_ibn fastreid --max-frames 30 --max-samples 500

# полный прогон
!python scripts/run_reid_eval.py --model osnet resnet50 resnet50_ibn fastreid

Already up to date.
> /usr/bin/python3 -m pip install -q --timeout 300 scikit-learn tabulate termcolor yacs prettytable easydict
> /usr/bin/python3 -m pip install -q --timeout 300 Cython h5py Pillow six scipy opencv-python matplotlib future yacs gdown imageio chardet
> /usr/bin/python3 -m pip install -q --timeout 300 --no-deps git+https://github.com/KaiyangZhou/deep-person-reid.git
  Preparing metadata (setup.py) ... done
> /usr/bin/python3 -c import torchreid
> git clone --depth 1 --branch v1.3.0 https://github.com/JDAI-CV/fast-reid.git /content/DeepSORT_Project_CV/third_party/fast_reid
Cloning into '/content/DeepSORT_Project_CV/third_party/fast_reid'...
remote: Enumerating objects: 536, done.
remote: Counting objects: 100% (536/536), done.
remote: Compressing objects: 100% (447/447), done.
remote: Total 536 (delta 109), reused 234 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (536/536), 2.15 MiB | 6.51 MiB/s, done.
Resolving deltas: 100% (109/109), done.
Note: switching 

## Модифицированный DeepSORT (детектор + ReID) → HOTA

Live-пайплайн: детектор → ReID-дескрипторы → ядро DeepSORT (Kalman + matching cascade) → MOT-txt.


In [10]:
NAME = "yolo_osnet"
# Прогон трекера по 6 видео (печатает FPS по каждому).
!python scripts/run_tracker.py --detector yolo --reid osnet --tracker-name {NAME}

# HOTA модифицированного трекера
!python scripts/run_eval.py --tracker-name {NAME} --results-dir results/tracking/{NAME}

Detector: yolo | ReID: osnet | mask_background: False | identity: False | tracker_name: yolo_osnet
/usr/local/lib/python3.12/dist-packages/torchreid/models/osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issu

## Перебор связок (detector × ReID) → сводная таблица

Несколько комбинаций, где считается HOTA по каждому видео + средний FPS, сравнивает с baseline и отмечает лучшую real-time связку (≥5 FPS)

In [11]:
!python scripts/run_sweep.py --combos yolo:osnet yolo:resnet50_ibn nanodet:osnet mmdet:osnet yolo_seg:osnet:seg

import pandas as pd
pd.read_csv("results/tracking/sweep_summary.csv")

Evaluating baseline (original) ...

CLEAR Config:
METRICS              : ['HOTA', 'CLEAR', 'Identity'] 
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Identity Config:
METRICS              : ['HOTA', 'CLEAR', 'Identity'] 
THRESHOLD            : 0.5                           
PRINT_CONFIG         : True                          

Evaluating 1 tracker(s) on 4 sequence(s) for 1 class(es) on MotChallenge2DBox dataset using the following metrics: HOTA, CLEAR, Identity, Count


Evaluating original

1 eval_sequence(KITTI-17, original)                                      0.0920 sec
2 eval_sequence(PETS09-S2L1, original)                                   0.6178 sec
3 eval_sequence(TUD-Campus, original)                                    0.0956 sec
4 eval_sequence(TUD-Stadtmitte, original)                                0.2173 sec

All sequences for original finished in 1.02 seconds

HOTA: original-pedestrian          HOTA      DetA 

,combo,TUD-Campus,TUD-Stadtmitte,KITTI-17,PETS09-S2L1,MOT16-09,MOT16-11,MEAN,mean_fps,beats_all
0,original,39.86,36.75,43.41,44.84,36.24,39.95,40.17,NaN,NaN
1,yolo_osnet,47.96,55.11,55.71,52.43,44.23,51.95,51.23,16.03,True
2,yolo_resnet50_ibn,40.02,53.38,57.14,52.41,38.80,44.48,47.70,14.64,True
3,nanodet_osnet,38.85,46.34,34.17,44.12,32.54,41.75,39.63,9.97,False
4,mmdet_osnet,38.24,57.32,44.47,48.83,40.06,48.95,46.31,6.26,False
5,yolo_seg_osnet_seg,48.99,55.30,57.93,55.78,38.82,49.69,51.08,10.60,True


## Оверлеи (original + best)

Headless-рендер через `cv2.VideoWriter` (без `cv2.imshow`, работает в Colab):
- `overlays/original/` — baseline DeepSORT
- `overlays/best/` — лучшая связка (поменяйте `NAME`, если другая)

In [12]:
NAME = "yolo_osnet"

# Headless render (no cv2.imshow) — original baseline + best combo.
!python scripts/run_overlays.py --convert_h264
!python scripts/run_overlays.py \
    --results-dir results/tracking/{NAME} \
    --overlays-dir overlays/best \
    --convert_h264

import os
print("overlays/original:", sorted(os.listdir("overlays/original")))
print("overlays/best:", sorted(os.listdir("overlays/best")))

Rendering overlays from results/baseline/original -> overlays/original
Saving KITTI-17 -> overlays/original/KITTI-17.avi
Saving PETS09-S2L1 -> overlays/original/PETS09-S2L1.avi
Saving TUD-Campus -> overlays/original/TUD-Campus.avi
Saving TUD-Stadtmitte -> overlays/original/TUD-Stadtmitte.avi
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-l

## Этап 6 — сегментация → маски убирают фон перед ReID

- `yolo_seg` — основной real-time вариант, **setup не нужен**
- `detectron2_seg` / `smp_seg` — optional quality mode (`setup_segmentation_colab.py`)

С флагом `--mask-background` фон зануляется до ReID.

In [13]:
# yolo_seg работает без detectron2; optional quality backends ниже
NAME = "yolo_seg_osnet_seg"
!python scripts/run_tracker.py --detector yolo_seg --reid osnet --mask-background --tracker-name {NAME}
!python scripts/run_eval.py --tracker-name {NAME} --results-dir results/tracking/{NAME}

# optional: detectron2 (сборка ~5–10 мин) + smp weights
# !python scripts/setup_segmentation_colab.py
# !python scripts/setup_segmentation_colab.py --download-smp-weights
# !python scripts/run_tracker.py --detector detectron2_seg --reid osnet --mask-background --tracker-name detectron2_seg_osnet_seg

Detector: yolo_seg | ReID: osnet | mask_background: True | identity: False | tracker_name: yolo_seg_osnet_seg
/usr/local/lib/python3.12/dist-packages/torchreid/models/osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub f

## Этап 7 — Additional: автономная база идентичностей (body ReID)

Онлайн-галерея (`IdentityDatabase`) + kNN lookup + majority-vote по окну + conflict resolution.
Встроена в live-трекер: `--identity` (sidecar `*_identity.csv`: track_id → identity).
Оценка на GT-кропах: Fowlkes–Mallows / ARI / V-measure для `db_raw` и `resolved`.
Параметры: `configs/identity.yaml` или CLI `--radius --k --representation --window`.

In [14]:
# Standalone eval на GT-кропах (defaults from configs/identity.yaml).
!python scripts/download_reid_models.py
!python scripts/run_identity_eval.py --reid osnet

# Live-трекер с identity DB (лучшая real-time связка + --identity)
NAME = "yolo_osnet_id"
!python scripts/run_tracker.py --detector yolo --reid osnet --identity --tracker-name {NAME}
# Sidecar: results/tracking/{NAME}/<seq>_identity.csv

# kNN-режим галереи (ablation)
!python scripts/run_identity_eval.py --reid osnet --representation knn --k 5 --radius 0.35

OK (exists): /content/DeepSORT_Project_CV/resources/models/fastreid/market_bot_R50.pth
OK (exists): /content/DeepSORT_Project_CV/resources/models/fastreid/market_bot_R50-ibn.pth
OK (exists): /content/DeepSORT_Project_CV/resources/models/torchreid/market1501_resnet50.pth
Done. osnet (torchreid) downloads weights on first run; resnet50 uses resources/models/torchreid/market1501_resnet50.pth; resnet50_ibn/fastreid use fast-reid checkpoints.
ReID: osnet | params: IdentityParams(radius=0.4, k=1, representation='centroid', window=30, conflict_policy='distance', max_per_identity=50)
/usr/local/lib/python3.12/dist-packages/torchreid/models/osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). 

# Эволюция параметров, per-video, GT-bbox ReID

- sweep по параметрам трекера (`max_cosine_distance`, `min_confidence`) → HOTA + FPS;
- sweep по параметрам identity DB (`radius`, `representation`) → FM/ARI;
- финальный прогон лучшей связки с **per-video** параметрами (`configs/tracker_params.yaml`);
- оценка ReID на **GT-боксах** (детектор выключен) → HOTA, изолируем качество ReID/ассоциации.

In [15]:
!git pull

# Эволюция параметров трекера (HOTA по каждому видео + FPS) на лучшей связке yolo+osnet
!python scripts/run_param_sweep.py --param max_cosine_distance --values 0.1 0.2 0.3 0.4 --detector yolo --reid osnet
!python scripts/run_param_sweep.py --param min_confidence --values 0.2 0.3 0.4 0.5 --detector yolo --reid osnet

import pandas as pd
display(pd.read_csv("results/param_sweep/sweep_max_cosine_distance.csv"))
display(pd.read_csv("results/param_sweep/sweep_min_confidence.csv"))

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 2.69 KiB | 919.00 KiB/s, done.
From https://github.com/Valeriia-Reznik-Dev/DeepSORT_Project_CV
   3b948d2..262a487  master     -> origin/master
Updating 3b948d2..262a487
Fast-forward
 notebooks/baseline_colab.ipynb  |  1 +
 reid/torchreid_ext.py           | 94 ++++++++++++++++++++++++++++++++++++++++-
 scripts/download_reid_models.py | 69 +++++++++++++++++++++++++-----
 3 files changed, 152 insertions(+), 12 deletions(-)
Loading yolo + osnet ...
/usr/local/lib/python3.12/dist-packages/torchreid/models/osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpicklin

,max_cosine_distance,TUD-Campus,TUD-Stadtmitte,KITTI-17,PETS09-S2L1,MOT16-09,MOT16-11,MEAN,mean_fps
0,0.1,44.17,59.13,54.77,45.50,42.51,47.58,48.95,17.22
1,0.2,47.96,55.11,55.71,52.43,44.23,51.95,51.23,16.99
2,0.3,43.50,51.29,56.58,52.53,39.84,47.62,48.56,16.25
3,0.4,43.50,51.29,56.50,52.54,39.12,44.21,47.86,16.52


,min_confidence,TUD-Campus,TUD-Stadtmitte,KITTI-17,PETS09-S2L1,MOT16-09,MOT16-11,MEAN,mean_fps
0,0.2,47.96,55.11,55.71,52.43,44.23,51.95,51.23,16.59
1,0.3,47.96,55.11,55.71,52.43,44.23,51.95,51.23,16.68
2,0.4,52.96,54.76,57.68,51.68,45.48,53.01,52.60,17.54
3,0.5,46.26,52.47,55.76,53.34,44.86,51.35,50.67,18.67


In [16]:
# Эволюция параметров identity DB (FM/ARI на GT-кропах)
!python scripts/run_identity_sweep.py --param radius --values 0.2 0.25 0.3 0.35 0.4 --reid osnet
!python scripts/run_identity_sweep.py --param representation --values centroid knn --reid osnet

import pandas as pd
display(pd.read_csv("results/identity_sweep/sweep_radius.csv"))
display(pd.read_csv("results/identity_sweep/sweep_representation.csv"))

Loading ReID: osnet ...
/usr/local/lib/python3.12/dist-packages/torchreid/models/osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(cached_f

,radius,resolved_fm,resolved_ari,resolved_v,db_raw_fm,pred_ids,true_ids
0,0.20,0.638,0.581,0.782,0.548,167.8,23.3
1,0.25,0.641,0.590,0.782,0.544,187.2,23.3
2,0.30,0.655,0.608,0.794,0.553,185.7,23.3
3,0.35,0.650,0.602,0.789,0.547,173.8,23.3
4,0.40,0.657,0.611,0.793,0.547,172.7,23.3


,representation,resolved_fm,resolved_ari,resolved_v,db_raw_fm,pred_ids,true_ids
0,centroid,0.657,0.611,0.793,0.547,172.7,23.3
1,knn,0.616,0.571,0.749,0.583,255.8,23.3


In [17]:
# Финальный прогон лучшей связки с per-video параметрами (configs/tracker_params.yaml)
NAME = "yolo_osnet_pv"
!python scripts/run_tracker.py --detector yolo --reid osnet \
    --params-config configs/tracker_params.yaml --tracker-name {NAME}
!python scripts/run_eval.py --tracker-name {NAME} --results-dir results/tracking/{NAME}

Detector: yolo | ReID: osnet | mask_background: False | identity: False | tracker_name: yolo_osnet_pv
/usr/local/lib/python3.12/dist-packages/torchreid/models/osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any i

In [18]:
# Сравните ReID-модели по HOTA без влияния ошибок детектора.
for REID in ["osnet", "resnet50", "resnet50_ibn", "fastreid"]:
    !python scripts/run_tracker.py --gt-detections --reid {REID} --tracker-name gtdet_{REID}
    !python scripts/run_eval.py --tracker-name gtdet_{REID} --results-dir results/tracking/gtdet_{REID}

Detector: GT-boxes | ReID: osnet | mask_background: False | identity: False | tracker_name: gtdet_osnet
/usr/local/lib/python3.12/dist-packages/torchreid/models/osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any

In [19]:
!zip -r project_outputs.zip \
    results/baseline/original/ \
    results/tracking/ \
    results/detectors/ \
    results/reid/ \
    results/identity/ \
    results/param_sweep/ \
    results/identity_sweep/ \
    overlays/

from google.colab import files
files.download("project_outputs.zip")


	zip warning: name not matched: results/reid/
  adding: results/baseline/original/ (stored 0%)
  adding: results/baseline/original/TUD-Campus.txt (deflated 65%)
  adding: results/baseline/original/TUD-Stadtmitte.txt (deflated 68%)
  adding: results/baseline/original/MOT16-09.txt (deflated 68%)
  adding: results/baseline/original/MOT16-11.txt (deflated 68%)
  adding: results/baseline/original/KITTI-17.txt (deflated 66%)
  adding: results/baseline/original/PETS09-S2L1.txt (deflated 70%)
  adding: results/tracking/ (stored 0%)
  adding: results/tracking/gtdet_osnet/ (stored 0%)
  adding: results/tracking/gtdet_osnet/TUD-Campus.txt (deflated 65%)
  adding: results/tracking/gtdet_osnet/TUD-Stadtmitte.txt (deflated 70%)
  adding: results/tracking/gtdet_osnet/MOT16-09.txt (deflated 68%)
  adding: results/tracking/gtdet_osnet/MOT16-11.txt (deflated 68%)
  adding: results/tracking/gtdet_osnet/KITTI-17.txt (deflated 66%)
  adding: results/tracking/gtdet_osnet/PETS09-S2L1.txt (deflated 68%)
  add

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>